# Emotion Detection - YOLO11 Training via Colab Extension

This notebook is configured to run using your IDE's **Colab Extension** GPU resources.

### ⚠️ Hardware Requirement:
**Note:** YOLO11 is optimized for **NVIDIA GPUs (CUDA)**. It does not support TPUs. 
Please go to **Runtime > Change runtime type** and ensure **T4 GPU** is selected.

In [ ]:
!pip install -U ultralytics kagglehub

In [ ]:
import os
import kagglehub

# Using the token you provided
os.environ['KAGGLE_API_TOKEN'] = 'Kaggle API token'
print("Kaggle API Token configured.")

In [ ]:
print("Downloading dataset...")
path = kagglehub.dataset_download("aklimarimi/8-facial-expressions-for-yolo")
print("Path to dataset files:", path)

In [ ]:
import yaml
import os
import kagglehub
import torch

# Safe path detection
try:
    data_root = path
except NameError:
    print("Variable 'path' not found. Re-retrieving dataset path...")
    data_root = kagglehub.dataset_download("aklimarimi/8-facial-expressions-for-yolo")

# Handle nested directory versioning
for root, dirs, files in os.walk(data_root):
    if 'data.yaml' in files:
        data_root = root
        break

config = {
    'path': data_root,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 9,
    'names': ['angry', 'contempt', 'disgust', 'fear', 'happy', 'natural', 'sad', 'sleepy', 'surprised']
}

with open('emotion_data.yaml', 'w') as f:
    yaml.dump(config, f)

print(f"emotion_data.yaml created with path: {data_root}")

# Automatic device selection and warnings
if torch.cuda.is_available():
    device = 0
    print("✅ GPU detected! Training will be high-speed. 🚀")
else:
    device = 'cpu'
    print("⚠️ NO GPU DETECTED.")
    print("IMPORTANT: If you are using a TPU runtime, please switch to a GPU runtime.")
    print("Go to: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")

In [ ]:
from ultralytics import YOLO

# Load YOLO11-Small (yolo11s.pt)
model = YOLO('yolo11s.pt')

# Train the model
results = model.train(
    data='emotion_data.yaml',
    epochs=50,
    imgsz=640,
    batch=16 if device == 0 else 4,
    device=device,
    name='emotion_yolo11s'
)